# 00 · Setup
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mundmina/dl_project/blob/main/notebooks/00_setup.ipynb)

Проверка окружения. **Runtime → Change runtime type → T4 GPU**, затем *Run all*.

Первый запуск скачивает Oxford-IIIT Pet (~800 MB) и Places365 val 256px (~500 MB) и кладёт архивы в
`MyDrive/bgbias/data/`. Следующие запуски берут архивы из Drive.

In [ ]:
# Shared setup: Drive, repo, dependencies. Same cell in every notebook.
# Colab: nothing to set. Local machine (own GPU): see LOCAL.md — open this notebook from the repo's notebooks/
# folder and set BGBIAS_DRIVE, BGBIAS_DATA and GITHUB_TOKEN before starting Jupyter.
REPO_URL = "https://github.com/mundmina/dl_project.git"
BRANCH = "main"

import os, subprocess, sys
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_DIR = "/content/dl_project"
    drive.mount("/content/drive")
    if not os.path.exists(REPO_DIR):
        subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, REPO_DIR], check=True)
    else:  # always run the latest code from GitHub
        subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "checkout", "-B", BRANCH, f"origin/{BRANCH}"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/requirements.txt"], check=True)
else:
    REPO_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))   # notebook opened from <repo>/notebooks
    assert os.path.exists(os.path.join(REPO_DIR, "src", "bgbias")), "open the notebook from the repo's notebooks/ folder"
    missing = [v for v in ("BGBIAS_DRIVE", "BGBIAS_DATA") if not os.environ.get(v)]
    assert not missing, f"set {missing} before starting Jupyter (see LOCAL.md)"
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", BRANCH], check=True)
commit = subprocess.run(["git", "-C", REPO_DIR, "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout
print("code version:", commit)
if f"{REPO_DIR}/src" not in sys.path:
    sys.path.insert(0, f"{REPO_DIR}/src")
# DataLoader workers on Windows start fresh interpreters: they find bgbias through PYTHONPATH, not sys.path
os.environ["PYTHONPATH"] = os.pathsep.join(p for p in (f"{REPO_DIR}/src", os.environ.get("PYTHONPATH")) if p)
try:  # ask for secret access now, not at the end of a long run (the prompt times out unattended)
    if not os.environ.get("GITHUB_TOKEN"):
        from google.colab import userdata
        userdata.get("GITHUB_TOKEN")
    print("GITHUB_TOKEN: ok")
except Exception as e:  # noqa: BLE001
    print(f"GITHUB_TOKEN not available ({type(e).__name__}): results will not be pushed to GitHub")

In [ ]:
import torch, torchvision
print("torch", torch.__version__, "| torchvision", torchvision.__version__)
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from bgbias.data_setup import prepare_pets, prepare_places
from bgbias.utils import DRIVE_ROOT, LOCAL_DATA

PET_ROOT = prepare_pets()        # /content/data/oxford-iiit-pet
PLACES_ROOT = prepare_places()   # /content/data/places365
OUT_DRIVE = DRIVE_ROOT / "week1"
OUT_DRIVE.mkdir(parents=True, exist_ok=True)
print(PET_ROOT, PLACES_ROOT)

In [ ]:
from pathlib import Path
n_img = len(list((PET_ROOT / "images").glob("*.jpg")))
n_tri = len([p for p in (PET_ROOT / "annotations" / "trimaps").glob("*.png") if not p.name.startswith("._")])
n_bg = len(list((PLACES_ROOT / "val_256").glob("*.jpg")))
print(f"pet images: {n_img}  trimaps: {n_tri}  places backgrounds: {n_bg}")
assert n_img > 7000 and n_tri > 7000 and n_bg == 36500

In [ ]:
import matplotlib.pyplot as plt
from bgbias import compose

image_id = "Abyssinian_1"
img = compose.load_image(PET_ROOT / "images" / f"{image_id}.jpg")
tri = compose.load_trimap(PET_ROOT / "annotations" / "trimaps" / f"{image_id}.png")
fig, ax = plt.subplots(1, 2, figsize=(8, 4))
ax[0].imshow(img); ax[1].imshow(tri, cmap="viridis")
for a in ax: a.axis("off")
plt.show()

Если всё прошло без ошибок — окружение готово, переходим к `01_data.ipynb`.